## **Object detection with YOLO (Ultralytics YOLOv8)**

Corrected version of the lecture notebook (original: `object_detection_original.ipynb`). Changes: the install cell is now a real command, the model name matches the comment, the cells are in a runnable order, the dataset path is defined once, the trained weights are found from the training result instead of a hard-coded `train5`, and mAP is computed with `model.val()`. Run it in Google Colab (GPU) or locally; Ultralytics must download weights on first use.

In [ ]:
!pip install -q ultralytics

In [ ]:
from pathlib import Path
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # nano: the smallest and fastest YOLOv8 model (3.2 M parameters)

### 1. Inference with a pretrained (COCO) model

In [ ]:
IMAGE = "my_image.jpg"       # put any image next to the notebook
results = model(IMAGE)
results[0].save("output.jpg")  # results[0].show() opens a window; in Colab use the saved file
results[0].boxes.xyxy.shape

In [ ]:
names = model.names
for box in results[0].boxes:
    print(names[int(box.cls)], round(float(box.conf), 2), box.xyxy.squeeze().tolist())

### 2. Dataset layout for training

```
custom_dataset/
├── images/{train,val}/  *.jpg
└── labels/{train,val}/  *.txt   # one line per object: class cx cy w h  (all normalised to [0, 1])
```

In [ ]:
DATA = Path("custom_dataset")
YAML = DATA / "data.yaml"
YAML.parent.mkdir(exist_ok=True)
# Edit `names` to the classes of YOUR dataset (the lecture example uses fruits; the dog/cat list is only an example)
YAML.write_text(f"path: {DATA.resolve()}\ntrain: images/train\nval: images/val\nnc: 2\nnames: ['dog', 'cat']\n")
print(YAML.read_text())

### 3. Training

In [ ]:
results = model.train(data=str(YAML), epochs=10, imgsz=416, batch=4)
best = Path(results.save_dir) / "weights" / "best.pt"   # no hard-coded runs/detect/train5
print(best)

### 4. Evaluation: precision, recall, mAP

`mAP50` is the mean over classes of the average precision at IoU threshold 0.5; `mAP50-95` averages it over thresholds 0.5, 0.55, ..., 0.95.

In [ ]:
trained = YOLO(best)
metrics = trained.val(data=str(YAML))
print("mAP50    =", round(metrics.box.map50, 3))
print("mAP50-95 =", round(metrics.box.map, 3))

### 5. Prediction with the trained model

In [ ]:
preds = trained.predict(source=str(DATA / "images" / "val"), save=True)   # images with boxes go to runs/detect/predict*
len(preds)

### 6. Webcam (optional, local computer only; does not work in Colab)

In [ ]:
# trained.predict(source=0, show=True)   # press q to stop